# EVE A3-155Ah — Calendar Aging (CaLT) Metadata Builder

## Source Data Gaps vs. Reference Calendar Aging Analysis

This notebook produces the same 31-column schema as `20260813_calendar_aging_analysis_exclusion_v4` but several columns cannot be populated due to differences between the EVE A3 source spreadsheet (`NF155L storage data (26.5.8)_translated.xlsx`) and the reference study's source data.

---

### Columns unavailable — missing from EVE A3 source

| Column | Reference calculation | Reason unavailable |
|---|---|---|
| Recovered charge capacity (Ah) | Measured per-period: ② Charge Capacity | No per-period recovered charge column in source |
| Recovered charge energy (Wh) | Measured per-period: ② Charge Energy | No per-period recovered charge energy in source |
| Coulombic efficiency | recovered\_dis\_cap / recovered\_chg\_cap | Requires recovered charge capacity |
| Energy efficiency | recovered\_dis\_energy / recovered\_chg\_energy | Requires recovered charge energy |
| Energy inefficiency | 1 − energy\_efficiency | Requires recovered charge energy |
| Relative energy efficiency | energy\_efficiency / baseline\_energy\_efficiency | Requires recovered charge energy |
| Relative energy inefficiency | energy\_inefficiency / baseline\_energy\_inefficiency | Requires recovered charge energy |

---

### Columns with approximated calculation

| Column | Reference calculation | EVE A3 approximation |
|---|---|---|
| Self discharge (mAh) | `shift(1) of SOC-adjustment charge − retained discharge` | `shift(1) of recovered discharge capacity − retained discharge` — no measured SOC adjustment charge column; `init_c` used for first period |
| Retained discharge capacity (Ah) | Measured at all SOC levels | Available at **100% SOC only** — NaN at 0% and 50% SOC (source col 11 not populated for those conditions) |

---

### Columns set to NAN in both reference and this notebook

Cycle, Time weighted deltaV (mV), Capacity weighted deltaV (mV), Relative time weighted deltaV, Relative capacity weighted deltaV, Ambient temperature (°C), Max charge temperature (°C), Max discharge temperature (°C), 10s DCIR 50%SOC (mOhms), SOC weighted 10s DCIR (mOhms), Relative 10s DCIR 50%SOC, Relative SOC weighted 10s DCIR, Thickness (mm), Relative thickness

---

### Source file unit note — capacity columns

The source file labels capacity columns (Before-test col 6, Residual col 11, Recovered col 13) as **"mAh"**, but all values are in **Ah** (e.g., 160.4 for a nominally 155 Ah cell). This is confirmed by the energy columns (col 7, 12, 14), which are correctly labeled Wh: 449 Wh ÷ 160 Ah ≈ 2.81 V average — physically consistent. No unit conversion is applied when reading these columns.

---

### Note — Relative discharge capacity > 1 at 0% SOC

For some cells stored at 0% SOC, `Rel_Dis_Cap` may slightly exceed 1.0 at early checkup periods. This is a measurement artefact: the before-test reference capacity (col 6) was measured at a 0.5P discharge rate, while the checkup recovered capacity (col 13) was measured at 0.25P. The lower C-rate yields a marginally higher measured capacity, causing the ratio to exceed unity for cells with minimal degradation. This does not indicate a physical capacity gain.

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print('Imports OK')

Imports OK


In [ ]:
# ============================================================
# CONFIG  – edit the items in this section only
# ============================================================
_INPUT_FILE  = r'C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260901_EVE-A3-155Ah_CyLT&CaLT\NF155L storage data (26.5.8)_translated.xlsx'
_OUTPUT_FILE = r'C:\Users\DanWindsor\OneDrive - Peak Energy\Documents\MCM_DataConversion\20260901_EVE-A3-155Ah_CyLT&CaLT\EVE-A3-155Ah_CaLT_MasterCode_v8.xlsx'

# Mapping: (source sheet name, temperature, average_soc)
# Order determines Cell ID assignment
_SHEET_MAP = [
    ('25\u2103-100%SOC Storage', 25, 1.0),
    ('25\u2103-50%SOC Storage',  25, 0.5),
    ('25\u2103-0%SOC Storage',   25, 0.0),
    ('45\u2103-100%SOC Storage', 45, 1.0),
    ('45\u2103-50%SOC Storage',  45, 0.5),
    ('45\u2103-0%SOC Storage',   45, 0.0),
    ('60\u2103-100%SOC Storage', 60, 1.0),
    ('60\u2103-50%SOC Storage',  60, 0.5),
    ('60\u2103-0%SOC Storage',   60, 0.0),
]

# Sheets requiring a 99.18% correction factor on relative capacity metrics.
# Applied to Rel_Ret_Cap and Rel_Dis_Cap per the source spreadsheet methodology.
_CAP_CORR       = 0.9918
_CAP_CORR_SHEETS = {
    '25\u2103-100%SOC Storage',
    '45\u2103-100%SOC Storage',
    '25\u2103-0%SOC Storage',
}
_CAP_CORR_NOTE = '99.18% correction factor applied to Rel_Ret_Cap and Rel_Dis_Cap per source spreadsheet methodology'

# Optional per-cell annotations keyed by serial number
_ANNOTATIONS = {
    # 'A9889': {'type_note': '', 'specific_notes': '', 'general_notes': ''},
}

print('Config OK')

In [12]:
# ============================================================
# STYLE CONSTANTS
# ============================================================
_DARK  = PatternFill('solid', fgColor='1F4E79')
_MED   = PatternFill('solid', fgColor='2E75B6')
_REQ   = PatternFill('solid', fgColor='D9E1F2')
_OPT   = PatternFill('solid', fgColor='E2EFDA')
_LGREY = PatternFill('solid', fgColor='F2F2F2')
_WHITE = PatternFill('solid', fgColor='FFFFFF')
_FW    = Font(name='Arial', bold=True, color='FFFFFF', size=10)
_FWB   = Font(name='Arial', bold=True, color='000000', size=10)
_FN    = Font(name='Arial', size=10)
_AC    = Alignment(horizontal='center', vertical='center', wrap_text=True)
_AL    = Alignment(horizontal='left',   vertical='center', wrap_text=True)
_AR    = Alignment(horizontal='right',  vertical='center')
_thin  = Side(style='thin', color='BFBFBF')
_BD    = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)

def _style(cell, fill=None, font=None, align=None, border=_BD):
    if fill:   cell.fill      = fill
    if font:   cell.font      = font
    if align:  cell.alignment = align
    if border is not None: cell.border = border

print('Styles OK')

Styles OK


In [ ]:
# ============================================================
# OUTPUT COLUMN DEFINITIONS
# (display header, internal DataFrame column name)
# ============================================================
# 31 columns match the reference schema (REFERENCE_COLUMNS_V3) exactly.
#
# NaN columns — no source data available (see markdown cell at top for details):
#   Cycle, Rec_Chg_Cap_Ah, CE, Rec_Chg_E_Wh, EE, EI, Rel_EE, Rel_EI,
#   Time_Wt_dV, Cap_Wt_dV, Rel_Time_Wt_dV, Rel_Cap_Wt_dV,
#   Amb_Temp, Max_Chg_Temp, Max_Dis_Temp,
#   DCIR_50SOC, DCIR_SOC_Wt, Rel_DCIR_50SOC, Rel_DCIR_SOC_Wt,
#   Thickness_mm, Rel_Thickness

_CD = [
    ('Time (days)',                           'Time_days'),
    ('Cycle',                                 'Cycle'),
    ('Retained discharge capacity (Ah)',      'Retained_Cap_Ah'),
    ('Recovered charge capacity (Ah)',        'Rec_Chg_Cap_Ah'),
    ('Recovered discharge capacity (Ah)',     'Rec_Dis_Cap_Ah'),
    ('Relative retained discharge capacity',  'Rel_Ret_Cap'),
    ('Relative discharge capacity',           'Rel_Dis_Cap'),
    ('Self discharge (mAh)',                  'Self_Dis_Ah'),
    ('Coulombic efficiency',                  'CE'),
    ('Retained discharge energy (Wh)',        'Retained_E_Wh'),
    ('Recovered charge energy (Wh)',          'Rec_Chg_E_Wh'),
    ('Recovered discharge energy (Wh)',       'Rec_Dis_E_Wh'),
    ('Relative retained discharge energy',    'Rel_Ret_E'),
    ('Relative discharge energy',             'Rel_Dis_E'),
    ('Energy efficiency',                     'EE'),
    ('Energy inefficiency',                   'EI'),
    ('Relative energy efficiency',            'Rel_EE'),
    ('Relative energy inefficiency',          'Rel_EI'),
    ('Time weighted deltaV (mV)',             'Time_Wt_dV'),
    ('Capacity weighted deltaV (mV)',         'Cap_Wt_dV'),
    ('Relative time weighted deltaV',         'Rel_Time_Wt_dV'),
    ('Relative capacity weighted deltaV',     'Rel_Cap_Wt_dV'),
    ('Ambient temperature (°C)',         'Amb_Temp'),
    ('Max charge temperature (°C)',      'Max_Chg_Temp'),
    ('Max discharge temperature (°C)',   'Max_Dis_Temp'),
    ('10s DCIR 50%SOC (mOhms)',              'DCIR_50SOC'),
    ('SOC weighted 10s DCIR (mOhms)',         'DCIR_SOC_Wt'),
    ('Relative 10s DCIR 50%SOC',             'Rel_DCIR_50SOC'),
    ('Relative SOC weighted 10s DCIR',        'Rel_DCIR_SOC_Wt'),
    ('Thickness (mm)',                         'Thickness_mm'),
    ('Relative thickness',                    'Rel_Thickness'),
]
_HEADERS = [h for h, _ in _CD]
_SRCCOLS = [s for _, s in _CD]
print(f'Column definitions OK  ({len(_CD)} columns)')

In [14]:
# ============================================================
# TIME LABEL → DAYS CONVERSION
# ============================================================
_DAYS_PER_MONTH = 365.25 / 12  # ~30.44

def _time_to_days(label):
    """'7d*N' → 7N days;  'NM' → N × 30.44 days."""
    if label is None:
        return np.nan
    label = str(label).strip()
    if label.lower().startswith('7d*'):
        try:
            return 7 * int(label.split('*')[1])
        except (IndexError, ValueError):
            return np.nan
    if label.upper().endswith('M'):
        try:
            return int(label[:-1]) * _DAYS_PER_MONTH
        except ValueError:
            return np.nan
    return np.nan

assert _time_to_days('7d*1')  == 7
assert _time_to_days('7d*4')  == 28
assert abs(_time_to_days('1M')  - 30.44) < 0.01
assert abs(_time_to_days('12M') - 365.25) < 0.01
print('Time conversion OK')

Time conversion OK


In [ ]:
# ============================================================
# DATA PARSER  – reads one source sheet, returns {serial: df}
# ============================================================

def _safe_float(v):
    if v is None or v == '/' or v == '':
        return np.nan
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan


def _parse_sheet(wb, sheet_name, cap_corr=1.0):
    """
    Source layout (0-indexed columns):
      col 1  : period label  (1M, 2M, …) – forward-filled per group
      col 2  : serial number
      col 6  : Before-test Capacity — first period only. Source labels 'mAh'; actual unit Ah.
      col 7  : Before-test Energy (Wh) — first period only
      col 11 : Residual/retained capacity. Source labels 'mAh'; actual unit Ah. NaN for 0%/50% SOC.
      col 12 : Residual/retained energy (Wh)
      col 13 : Recovered discharge capacity. Source labels 'mAh'; actual unit Ah.
      col 14 : Recovered discharge energy (Wh)

    All three capacity columns (6, 11, 13) are mislabeled 'mAh' in the source; values are in Ah.
    No /1000 conversion is applied. Energy columns (7, 12, 14) are correctly labeled Wh.

    cap_corr: correction factor applied to Rel_Ret_Cap and Rel_Dis_Cap (default 1.0 = no correction).
    Pass 0.9918 for sheets that require the source spreadsheet's 99.18% capacity correction.
    The Day 0 synthetic baseline row always has Rel_Ret_Cap = Rel_Dis_Cap = 1.0 regardless.

    NOTE: No per-period recovered CHARGE data exists in the source.
    Rec_Chg_Cap_Ah, Rec_Chg_E_Wh, CE, EE, EI, Rel_EE, Rel_EI are NaN.
    """
    ws   = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))

    C_PERIOD  = 1
    C_SERIAL  = 2
    C_CAP_BT  = 6   # Before-test Capacity (Ah) — first period only; source labels mAh
    C_E_BT    = 7   # Before-test Energy (Wh) — first period only
    C_RET_C   = 11  # Residual/retained capacity (Ah); source labels mAh; NaN for 0%/50% SOC
    C_RET_E   = 12  # Residual/retained energy (Wh)
    C_REC_C   = 13  # Recovered discharge capacity (Ah); source labels mAh
    C_REC_E   = 14  # Recovered discharge energy (Wh)

    serials = []
    for r in rows:
        s = r[C_SERIAL] if len(r) > C_SERIAL else None
        if s and isinstance(s, str) and s not in ('No.', 'Data') and s not in serials:
            serials.append(s)

    records     = {s: [] for s in serials}
    initial_cap = {}
    initial_e   = {}
    cur_period  = None

    for r in rows:
        if len(r) <= C_SERIAL:
            continue
        serial = r[C_SERIAL]
        if not serial or not isinstance(serial, str) or serial in ('No.', 'Data'):
            continue
        if serial not in serials:
            continue

        p = r[C_PERIOD]
        if p and str(p).strip() not in ('Data', ''):
            cur_period = str(p).strip()
        if cur_period is None:
            continue

        cap_bt = _safe_float(r[C_CAP_BT] if len(r) > C_CAP_BT else None)
        e_bt   = _safe_float(r[C_E_BT]   if len(r) > C_E_BT   else None)

        if serial not in initial_cap and not np.isnan(cap_bt):
            initial_cap[serial] = cap_bt  # Ah
            initial_e[serial]   = e_bt    # Wh

        records[serial].append({
            'Period':    cur_period,
            'Time_days': _time_to_days(cur_period),
            # Source labels these columns 'mAh'; actual unit is Ah
            'ret_c_Ah':  _safe_float(r[C_RET_C] if len(r) > C_RET_C else None),
            'ret_e_Wh':  _safe_float(r[C_RET_E] if len(r) > C_RET_E else None),
            'rec_c_Ah':  _safe_float(r[C_REC_C] if len(r) > C_REC_C else None),
            'rec_e_Wh':  _safe_float(r[C_REC_E] if len(r) > C_REC_E else None),
        })

    result = {}
    for serial in serials:
        raw = pd.DataFrame(records[serial])
        if raw.empty:
            result[serial] = raw
            continue

        init_c = initial_cap.get(serial, np.nan)  # Ah
        init_e = initial_e.get(serial, np.nan)    # Wh

        df = pd.DataFrame()
        df['Time_days']       = raw['Time_days']
        df['Cycle']           = np.nan
        # Capacity values already in Ah — no /1000 conversion needed
        df['Retained_Cap_Ah'] = raw['ret_c_Ah']
        df['Rec_Chg_Cap_Ah']  = np.nan
        df['Rec_Dis_Cap_Ah']  = raw['rec_c_Ah']
        # cap_corr applies the 99.18% correction where required by the source methodology
        df['Rel_Ret_Cap']     = np.where(init_c > 0, raw['ret_c_Ah'] / init_c * cap_corr, np.nan)
        df['Rel_Dis_Cap']     = np.where(init_c > 0, raw['rec_c_Ah'] / init_c * cap_corr, np.nan)

        # Self-discharge in mAh — mirrors reference formula structure:
        # shift(1) of recharge amount − retained discharge.
        # Approximated with shift(1) of recovered discharge capacity; init_c for first period.
        # Source capacity values are in Ah; multiply by 1000 to output mAh.
        prev_rec_c = raw['rec_c_Ah'].shift(1).fillna(init_c)
        df['Self_Dis_Ah'] = np.where(
            ~np.isnan(raw['ret_c_Ah'].values),
            (prev_rec_c - raw['ret_c_Ah']) * 1000.0,
            np.nan)

        df['CE']             = np.nan
        df['Retained_E_Wh']  = raw['ret_e_Wh']
        df['Rec_Chg_E_Wh']   = np.nan
        df['Rec_Dis_E_Wh']   = raw['rec_e_Wh']
        df['Rel_Ret_E']       = np.where(init_e > 0, raw['ret_e_Wh'] / init_e, np.nan)
        df['Rel_Dis_E']       = np.where(init_e > 0, raw['rec_e_Wh'] / init_e, np.nan)
        df['EE']              = np.nan
        df['EI']              = np.nan
        df['Rel_EE']          = np.nan
        df['Rel_EI']          = np.nan
        df['Time_Wt_dV']      = np.nan
        df['Cap_Wt_dV']       = np.nan
        df['Rel_Time_Wt_dV']  = np.nan
        df['Rel_Cap_Wt_dV']   = np.nan
        df['Amb_Temp']         = np.nan
        df['Max_Chg_Temp']    = np.nan
        df['Max_Dis_Temp']    = np.nan
        df['DCIR_50SOC']      = np.nan
        df['DCIR_SOC_Wt']     = np.nan
        df['Rel_DCIR_50SOC']  = np.nan
        df['Rel_DCIR_SOC_Wt'] = np.nan
        df['Thickness_mm']    = np.nan
        df['Rel_Thickness']   = np.nan

        # Synthetic Day 0 baseline row: all relative metrics = 1.0, self-discharge = 0 mAh.
        # Day 0 is a theoretical reference point; cap_corr is not applied here.
        # init_c is in Ah; Retained_Cap_Ah and Rec_Dis_Cap_Ah are Ah values.
        day0 = {col: np.nan for col in df.columns}
        day0['Time_days']       = 0.0
        day0['Retained_Cap_Ah'] = init_c if not np.isnan(init_c) else np.nan
        day0['Rec_Dis_Cap_Ah']  = init_c if not np.isnan(init_c) else np.nan
        day0['Rel_Ret_Cap']     = 1.0
        day0['Rel_Dis_Cap']     = 1.0
        day0['Self_Dis_Ah']     = 0.0
        day0['Retained_E_Wh']   = init_e if not np.isnan(init_e) else np.nan
        day0['Rec_Dis_E_Wh']    = init_e if not np.isnan(init_e) else np.nan
        day0['Rel_Ret_E']       = 1.0
        day0['Rel_Dis_E']       = 1.0
        df = pd.concat([pd.DataFrame([day0]), df], ignore_index=True)

        result[serial] = df

    return result


print('Parser defined')

In [16]:
# ============================================================
# METADATA SHEET BUILDER
# ============================================================

def _meta_sheet(ws, cells_cfg):
    ws.title = 'Metadata'
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'B3'

    _REQ_FIELDS = [
        ('Cell ID',                 'cell_num'),
        ('Temperature',             'temperature'),
        ('Test Type',               'test_type'),
        ('Calendar:\nAverage SOC',  'avg_soc'),
        ('Cycling:\nP-rate',        'p_rate'),
        ('Serial number',           'serial'),
    ]
    _OPT_FIELDS = [
        ('Type note',        'type_note'),
        ('Specific\nnotes',  'specific_notes'),
        ('General\nnotes',   'general_notes'),
    ]
    _ALL = _REQ_FIELDS + _OPT_FIELDS
    N_REQ = len(_REQ_FIELDS)
    N_ALL = len(_ALL)

    # Row 1: two-band merged header
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=N_REQ)
    c = ws.cell(1, 1, 'Required fields for calendar fade model')
    _style(c, fill=_REQ, font=_FWB, align=_AC)

    ws.merge_cells(start_row=1, start_column=N_REQ+1, end_row=1, end_column=N_ALL)
    c = ws.cell(1, N_REQ+1, 'All other fields optional')
    _style(c, fill=_OPT, font=_FWB, align=_AC)

    # Row 2: column headers
    for ci, (label, _) in enumerate(_ALL, start=1):
        hc = ws.cell(2, ci, label)
        _style(hc, fill=_MED, font=_FW, align=_AC)

    # Data rows
    for ri, cfg in enumerate(cells_cfg, start=3):
        fill = _LGREY if (ri % 2 == 1) else _WHITE
        for ci, (_, key) in enumerate(_ALL, start=1):
            val = cfg.get(key, '')
            dc  = ws.cell(ri, ci, val)
            _style(dc, fill=fill, font=_FN, align=_AC)

    col_widths = [8, 12, 10, 14, 12, 16, 12, 16, 16]
    for ci, w in enumerate(col_widths, start=1):
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[1].height = 18
    ws.row_dimensions[2].height = 36

print('Metadata builder defined')

Metadata builder defined


In [17]:
# ============================================================
# PER-CELL SHEET BUILDER
# ============================================================

def _cell_sheet(ws, label, df):
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'B3'

    # Row 1: merged title banner
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(_CD))
    h = ws.cell(1, 1, label)
    _style(h, fill=_DARK, font=_FW, align=_AC)

    # Row 2: column headers
    for ci, hdr in enumerate(_HEADERS, start=1):
        hc = ws.cell(2, ci, hdr)
        _style(hc, fill=_MED, font=_FW, align=_AC)

    # Data rows
    for ri, row in df.reset_index(drop=True).iterrows():
        fill = _LGREY if (ri % 2 == 0) else _WHITE
        for ci, src in enumerate(_SRCCOLS, start=1):
            val = row[src] if src in df.columns else 'NAN'
            if isinstance(val, float) and np.isnan(val):
                val = ''
            dc = ws.cell(ri + 3, ci, val)
            _style(dc, fill=fill, font=_FN, align=_AL if ci == 1 else _AR)

    col_widths = [12] + [26] * (len(_CD) - 1)
    for ci, w in enumerate(col_widths, start=1):
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[1].height = 20
    ws.row_dimensions[2].height = 40

print('Cell sheet builder defined')

Cell sheet builder defined


In [ ]:
# ============================================================
# MAIN RUN
# ============================================================
print(f'Reading: {_INPUT_FILE}')
wb_in = openpyxl.load_workbook(_INPUT_FILE, data_only=True)

all_cells = []
for sheet_name, temperature, avg_soc in _SHEET_MAP:
    if sheet_name not in wb_in.sheetnames:
        print(f'  WARNING: sheet not found: {sheet_name!r}')
        continue
    corr = _CAP_CORR if sheet_name in _CAP_CORR_SHEETS else 1.0
    note = _CAP_CORR_NOTE if sheet_name in _CAP_CORR_SHEETS else ''
    for serial, df in _parse_sheet(wb_in, sheet_name, cap_corr=corr).items():
        ann = _ANNOTATIONS.get(serial, {})
        all_cells.append({
            'serial':         serial,
            'temperature':    temperature,
            'avg_soc':        avg_soc,
            'test_type':      'CaLT',
            'p_rate':         0,
            'type_note':      ann.get('type_note', ''),
            'specific_notes': ann.get('specific_notes', '') or note,
            'general_notes':  ann.get('general_notes', ''),
            'df':             df,
        })

for idx, cfg in enumerate(all_cells, start=1):
    cfg['cell_num'] = idx

print(f'Parsed {len(all_cells)} cells across {len(_SHEET_MAP)} conditions')

wb_out = Workbook()
wb_out.remove(wb_out.active)

ws_meta = wb_out.create_sheet('Metadata')
_meta_sheet(ws_meta, all_cells)
print('  [Metadata] written')

for cfg in all_cells:
    cnum   = cfg['cell_num']
    serial = cfg['serial']
    df     = cfg['df']
    temp   = cfg['temperature']
    soc    = cfg['avg_soc']

    ws    = wb_out.create_sheet(f'Cell {cnum}')
    label = f'{serial}  |  {temp}°C  |  SOC {soc:.0%}'
    _cell_sheet(ws, label, df)

    n_pts     = len(df)
    last_cap  = df['Rel_Dis_Cap'].dropna()
    cap_val   = f'{last_cap.iloc[-1]:.4f}' if not last_cap.empty else 'n/a'
    last_ret  = df['Retained_Cap_Ah'].dropna()
    ret_val   = f'{last_ret.iloc[-1]:.3f} Ah' if not last_ret.empty else 'NaN'
    last_dis  = df['Self_Dis_Ah'].dropna()
    dis_val   = f'{last_dis.iloc[-1]:.1f} mAh' if not last_dis.empty else 'NaN'
    print(f'  [Cell {cnum:>2}]  {serial:<10}  {temp}°C  SOC={soc:.0%}  '
          f'{n_pts} pts  Rel_Dis_Cap={cap_val}  Retained={ret_val}  Self_Dis={dis_val}')

Path(_OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)
wb_out.save(_OUTPUT_FILE)
print(f'\nSaved: {_OUTPUT_FILE}')